# Track A Historia: RecurrentPPO vs DMLPA

Notebook final para revisar con David. La pregunta no es si una red tiene reward alto durante entrenamiento, sino si una arquitectura con historia mejora las metricas de resiliencia de Garrido frente a la mejor politica estatica bajo el mismo panel, riesgo y espacio de decision.

**Arquitecturas**
- `recurrent_ppo`: PPO recurrente real (`MlpLstmPolicy`). La memoria vive en el estado oculto LSTM.
- `dmlpa_ppo`: PPO normal con un extractor DMLPA/Transformer sobre `VecFrameStack(30)`. Ve una ventana reciente, pero no mantiene estado recurrente oculto.

**Espacios de decision**
- `thesis_factorized`: las variables discretas de Garrido (`I_{t,S}`, `S`).
- `continuous_it_s`: las mismas variables de Garrido, pero con buffer continuo.

**Riesgos**
- `severe_training`: maximo estres antes de guerra.
- `war_stress_v1`: extension explicita tipo guerra.

Este notebook asume entrenamiento serio en Kaggle GPU. Local se usa solo para smokes cortos.


## Criterio de victoria

No usar `reward_total` como criterio de victoria. El ranking usa:

1. `delta_ret_all_orders`
2. `delta_flow_fill`
3. `delta_fill`
4. `delta_ret_p10_all`

Regla de parada:
- Si todas las 8 configs pierden contra la mejor estatica por mas de `0.01` en fill y `0.02` en ReT/flow-fill, paramos Track A.
- Si alguna empata fill dentro de `0.01` y mejora ReT all-orders o flow-fill, pasa a confirmatorio.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd

SEED = 4242
REPO_URL = "https://github.com/Thom-320/scres-ia.git"
TARGET_REF = os.environ.get("SCRESIA_TARGET_REF", "codex/garrido-postfix-reruns")
REPO_DIR = Path(os.environ.get("SCRESIA_REPO_DIR", "/kaggle/working/scres-ia"))
OUTPUT_ROOT = Path(os.environ.get("SCRESIA_OUTPUT_ROOT", "/kaggle/working/track_a_history_outputs"))

# Safety switches. In Kaggle, set RUN_SCREENING=True after the smoke cells pass.
RUN_SMOKES = True
RUN_SCREENING = True
RUN_CONFIRMATORY = False

print({
    "target_ref": TARGET_REF,
    "repo_dir": str(REPO_DIR),
    "output_root": str(OUTPUT_ROOT),
})


In [ ]:
# GPU check. On Kaggle, this should print cuda_available=True and a GPU name.
try:
    import torch
    print("cuda_available:", torch.cuda.is_available())
    print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
except Exception as exc:
    print("Torch check failed:", repr(exc))


In [ ]:
def run(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)

if REPO_DIR.exists():
    print("Using existing repo", REPO_DIR)
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--depth", "1", "--branch", TARGET_REF, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)
run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


In [ ]:
# Contract audit: these flags must be present in every serious run.
FINAL_ENV_FLAGS = {
    "risk_occurrence_mode": "thesis_periodic",
    "raw_material_flow_mode": "kit_equivalent_order_up_to",
    "raw_material_order_up_to_multiplier": 2.0,
    "stochastic_pt": True,
    "stochastic_pt_spread": 1.0,
    "observation_version": "v5",
    "observation_mode": "env_sdm_history_reward",
    "reward_profile": "ret_ladder_steep",
}
FINAL_ENV_FLAGS


## Smoke tests cortos

Estos dos smokes son intencionalmente pequenos. Solo prueban que los dos caminos construyen modelo, entrenan, evaluan contra grilla estatica y escriben `policy_summary.csv`.


In [ ]:
def smoke_command(label: str, algo: str, action_space: str, risk: str, seed: int) -> list[str]:
    cmd = [
        sys.executable,
        "scripts/run_thesis_decision_ppo_smoke.py",
        "--label", label,
        "--output-root", str(OUTPUT_ROOT / "smokes"),
        "--train-timesteps", "1024",
        "--eval-episodes", "1",
        "--seed", str(seed),
        "--algo", algo,
        "--device", "auto",
        "--history-window", "30",
        "--risk-level", risk,
        "--risk-occurrence-mode", "thesis_periodic",
        "--raw-material-flow-mode", "kit_equivalent_order_up_to",
        "--raw-material-order-up-to-multiplier", "2.0",
        "--action-space-mode", action_space,
        "--inventory-period-mode", "thesis_strict",
        "--max-steps", "8",
        "--include-static-grid",
        "--no-eval-ai-on-garrido-cfis",
        "--n-steps", "64",
        "--batch-size", "32",
        "--n-epochs", "1",
        "--reward-mode", "ReT_ladder_v1",
        "--ret-ladder-w-sc", "0.80",
        "--ret-ladder-w-rc", "0.20",
        "--ret-ladder-w-ef", "0.00",
        "--ret-ladder-gate-beta", "20.0",
        "--stochastic-pt",
        "--stochastic-pt-spread", "1.0",
        "--train-cfis", "31",
        "--garrido-cfis", "31",
        "--train-risk-profile", risk,
        "--eval-risk-profile", risk,
    ]
    return cmd

smokes = [
    smoke_command("david_smoke_dmlpa_thesis_severe", "dmlpa_ppo", "thesis_factorized", "severe_training", 101),
    smoke_command("david_smoke_recurrent_continuous_war", "recurrent_ppo", "continuous_it_s", "war_stress_v1", 102),
]

if RUN_SMOKES:
    for cmd in smokes:
        run(cmd, cwd=REPO_DIR)
else:
    for cmd in smokes:
        print(" ".join(cmd))


## Screening serio: 8 configs

Matriz: `2 arquitecturas x 2 espacios x 2 riesgos`.

Este bloque es el que debe correr en Kaggle GPU. Con `100k` timesteps por config es screening, no confirmatorio.


In [ ]:
screening_cmd = [
    sys.executable,
    "scripts/run_track_a_exhaustion_sweep.py",
    "--label-prefix", "david_track_a_history_screen",
    "--output-root", str(OUTPUT_ROOT / "screening"),
    "--algos", "recurrent_ppo", "dmlpa_ppo",
    "--action-space-modes", "thesis_factorized", "continuous_it_s",
    "--reward-profiles", "ret_ladder_steep",
    "--risk-levels", "severe_training", "war_stress_v1",
    "--pt-profiles", "stoch_pt_hist",
    "--use-cf-risk-profile",
    "--panel-cfis", "31-90",
    "--train-timesteps", "100000",
    "--eval-episodes", "5",
    "--max-steps", "260",
    "--n-steps", "1024",
    "--batch-size", "64",
    "--n-epochs", "10",
    "--history-window", "30",
    "--device", "auto",
    "--stop-on-error",
]

if RUN_SCREENING:
    run(screening_cmd, cwd=REPO_DIR)
else:
    print(" ".join(screening_cmd))


In [ ]:
def latest_sweep_dir(root: Path) -> Path:
    candidates = sorted((root / "screening").glob("david_track_a_history_screen_*"))
    if not candidates:
        raise FileNotFoundError(f"No screening sweep found under {root / 'screening'}")
    return candidates[-1]

sweep_dir = latest_sweep_dir(OUTPUT_ROOT)
summary_path = sweep_dir / "sweep_summary.csv"
print("sweep_dir:", sweep_dir)
print("summary:", summary_path)
summary = pd.read_csv(summary_path)
summary


In [ ]:
# Ranking: no reward hacking. Sort by resilience metrics against same-panel best static.
rank_cols = [
    "algo",
    "action_space_mode",
    "risk_level",
    "status",
    "delta_ret_all_orders",
    "delta_flow_fill",
    "delta_fill",
    "delta_ret_p10_all",
    "delta_stockout_week_pct",
    "best_static_policy",
    "ppo_ret_all_orders",
    "best_static_ret_all_orders",
    "ppo_flow_fill",
    "best_static_flow_fill",
    "ppo_fill",
    "best_static_fill",
]
rank_cols = [c for c in rank_cols if c in summary.columns]
ranked = summary.sort_values(
    by=["delta_ret_all_orders", "delta_flow_fill", "delta_fill", "delta_ret_p10_all"],
    ascending=[False, False, False, False],
)
ranked[rank_cols].head(8)


In [ ]:
def promotion_decision(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    for col in ["delta_fill", "delta_ret_all_orders", "delta_flow_fill"]:
        if col not in work:
            work[col] = 0.0
    work["promote"] = (
        ((work["delta_fill"] >= -0.01) & ((work["delta_ret_all_orders"] > 0.0) | (work["delta_flow_fill"] > 0.0)))
        | (work["delta_fill"] >= 0.01)
        | (work["delta_ret_all_orders"] >= 0.02)
        | (work["delta_flow_fill"] >= 0.02)
    )
    work["stop_track_a_loss"] = (
        (work["delta_fill"] < -0.01)
        & (work["delta_ret_all_orders"] < -0.02)
        & (work["delta_flow_fill"] < -0.02)
    )
    return work

judged = promotion_decision(ranked)
print("all_runs_fail_stop_rule:", bool(judged["stop_track_a_loss"].all()))
print("promoted_count:", int(judged["promote"].sum()))
judged[[c for c in ["algo", "action_space_mode", "risk_level", "delta_ret_all_orders", "delta_flow_fill", "delta_fill", "promote", "stop_track_a_loss"] if c in judged.columns]].head(8)


## Confirmatorio top-2

Solo correr si el screening tiene al menos una config promovible. El confirmatorio son `500k x 3 seeds` para las dos mejores configs por `delta_ret_all_orders` y `delta_flow_fill`.


In [ ]:
def confirmatory_commands(df: pd.DataFrame, top_n: int = 2) -> list[list[str]]:
    promoted = df[df["promote"]].head(top_n)
    commands: list[list[str]] = []
    for _, row in promoted.iterrows():
        for seed in [4242, 4243, 4244]:
            commands.append([
                sys.executable,
                "scripts/run_track_a_exhaustion_sweep.py",
                "--label-prefix", f"confirm_{row['algo']}_{row['action_space_mode']}_{row['risk_level']}_seed{seed}",
                "--output-root", str(OUTPUT_ROOT / "confirmatory"),
                "--algos", str(row["algo"]),
                "--action-space-modes", str(row["action_space_mode"]),
                "--reward-profiles", "ret_ladder_steep",
                "--risk-levels", str(row["risk_level"]),
                "--pt-profiles", "stoch_pt_hist",
                "--use-cf-risk-profile",
                "--panel-cfis", "31-90",
                "--train-timesteps", "500000",
                "--eval-episodes", "10",
                "--seed", str(seed),
                "--max-steps", "260",
                "--n-steps", "1024",
                "--batch-size", "64",
                "--n-epochs", "10",
                "--history-window", "30",
                "--device", "auto",
                "--stop-on-error",
            ])
    return commands

commands = confirmatory_commands(judged)
for cmd in commands:
    print(" ".join(cmd))

if RUN_CONFIRMATORY:
    for cmd in commands:
        run(cmd, cwd=REPO_DIR)


In [ ]:
# Pack outputs for download from Kaggle.
archive_base = Path("/kaggle/working/track_a_history_outputs") if Path("/kaggle/working").exists() else Path("track_a_history_outputs")
if OUTPUT_ROOT.exists():
    archive = shutil.make_archive(str(archive_base), "zip", root_dir=str(OUTPUT_ROOT))
    print("archive:", archive)
else:
    print("No output root found yet:", OUTPUT_ROOT)


## Notes for David

- DMLPA aqui no reemplaza a RecurrentPPO: es PPO con un extractor Transformer sobre una ventana de 30 observaciones apiladas.
- RecurrentPPO usa LSTM y por eso no necesita `VecFrameStack`.
- Las notebooks viejas no son fuente final porque algunas no fijaban `thesis_periodic`, `kit_equivalent_order_up_to`, o entrenaban sobre el entorno crudo en vez del wrapper normalizado.
- Si DMLPA gana solo en reward pero no mejora `ret_mean_all_orders_zero_unfulfilled`, `flow_fill_rate`, `stockout_week_pct` o `ret_p10_all`, no cuenta como victoria.
